In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

import dice_ml
from dice_ml import Dice
from dice_ml.utils import helpers  # helper functions

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
dataset = helpers.load_adult_income_dataset().sample(5000)  # downsampling to reduce ML model fitting time
helpers.get_adult_data_info()

c:\Users\acbriza\anaconda3\envs\dpncf\Lib\site-packages\dice_ml\utils\helpers.py:70: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adult_data = adult_data.replace({'income': {'<=50K': 0, '>50K': 1}})


{'age': 'age',
 'workclass': 'type of industry (Government, Other/Unknown, Private, Self-Employed)',
 'education': 'education level (Assoc, Bachelors, Doctorate, HS-grad, Masters, Prof-school, School, Some-college)',
 'marital_status': 'marital status (Divorced, Married, Separated, Single, Widowed)',
 'occupation': 'occupation (Blue-Collar, Other/Unknown, Professional, Sales, Service, White-Collar)',
 'race': 'white or other race?',
 'gender': 'male or female?',
 'hours_per_week': 'total work hours per week',
 'income': '0 (<=50K) vs 1 (>50K)'}

In [5]:
target = dataset["income"]

# Split data into train and test
datasetX = dataset.drop("income", axis=1)
x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = ["age", "hours_per_week"]
categorical = x_train.columns.difference(numerical)

# We create the preprocessing pipelines for both numeric and categorical data.
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', categorical_transformer, categorical)])

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', RandomForestClassifier())])
model = clf.fit(x_train, y_train)

In [6]:
d = dice_ml.Data(dataframe=dataset, continuous_features=['age', 'hours_per_week'], outcome_name='income')
m = dice_ml.Model(model=model, backend="sklearn")

In [7]:
exp = Dice(d, m, method="random")
query_instance = x_train[0:50]
e1 = exp.generate_counterfactuals(query_instance, total_CFs=10, desired_range=None,
                                  desired_class="opposite",
                                  permitted_range=None, features_to_vary="all")


100%|██████████| 50/50 [00:13<00:00,  3.68it/s]


In [52]:
print([e.visualize_as_dataframe(show_only_changes=True) for e in e1.cf_examples_list[:10]])

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,50,Private,Bachelors,Married,Blue-Collar,White,Male,40,0



Diverse Counterfactual set (new outcome: 1.0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,-,-,Sales,Other,-,-,1
1,-,-,-,-,-,-,-,90.0,1
2,-,Other/Unknown,-,-,Service,-,-,-,1
3,65.0,-,-,-,-,-,-,56.0,1
4,45.0,-,-,-,-,-,-,-,1
5,-,-,Prof-school,-,-,-,-,47.0,1
6,62.0,-,-,-,Sales,-,-,-,1
7,-,Self-Employed,Prof-school,-,-,-,-,-,1
8,46.0,-,-,-,-,-,-,86.0,1
9,-,-,-,-,Other/Unknown,-,-,75.0,1


Query instance (original outcome : 0)


...

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,41,Private,HS-grad,Divorced,Service,White,Female,40,0



Diverse Counterfactual set (new outcome: 1.0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,72.0,-,-,Married,-,-,Male,-,1
1,-,Government,-,Married,Blue-Collar,-,-,-,1
2,66.0,-,-,Married,-,-,Male,-,1
3,-,-,-,Married,Sales,-,Male,-,1
4,-,Government,Bachelors,Married,-,-,-,-,1
5,45.0,-,-,-,Blue-Collar,-,-,-,-
6,-,-,Bachelors,Married,-,-,Male,-,1
7,85.0,-,-,-,-,-,Male,60.0,1
8,-,-,-,Married,-,Other,Male,-,1
9,-,-,-,Married,-,-,Male,-,1


[None, None, None, None, None, None, None, None, None, None]


In [43]:
imp = exp.local_feature_importance(query_instance, cf_examples_list=e1.cf_examples_list)
print(imp.local_importance)

[{'hours_per_week': 0.5, 'occupation': 0.4, 'age': 0.4, 'workclass': 0.2, 'education': 0.2, 'race': 0.1, 'marital_status': 0.0, 'gender': 0.0}, {'education': 1.0, 'age': 0.3, 'workclass': 0.2, 'gender': 0.2, 'occupation': 0.1, 'marital_status': 0.0, 'race': 0.0, 'hours_per_week': 0.0}, {'hours_per_week': 0.9, 'education': 0.4, 'race': 0.3, 'workclass': 0.2, 'marital_status': 0.2, 'occupation': 0.0, 'gender': 0.0, 'age': 0.0}, {'marital_status': 0.8, 'age': 0.4, 'education': 0.2, 'occupation': 0.2, 'race': 0.1, 'hours_per_week': 0.1, 'workclass': 0.0, 'gender': 0.0}, {'occupation': 0.4, 'age': 0.4, 'hours_per_week': 0.4, 'marital_status': 0.3, 'workclass': 0.1, 'education': 0.1, 'race': 0.0, 'gender': 0.0}, {'education': 0.8, 'workclass': 0.3, 'occupation': 0.3, 'gender': 0.2, 'hours_per_week': 0.2, 'age': 0.1, 'marital_status': 0.0, 'race': 0.0}, {'education': 0.3, 'marital_status': 0.3, 'occupation': 0.3, 'age': 0.3, 'hours_per_week': 0.3, 'workclass': 0.1, 'race': 0.1, 'gender': 0.0}

In [46]:
imp = exp.local_feature_importance(query_instance, posthoc_sparsity_param=None)
print(imp.local_importance)

100%|██████████| 50/50 [00:02<00:00, 19.13it/s]

[{'age': 0.7, 'occupation': 0.6, 'workclass': 0.2, 'hours_per_week': 0.2, 'race': 0.1, 'education': 0.0, 'marital_status': 0.0, 'gender': 0.0}, {'education': 0.8, 'age': 0.4, 'gender': 0.3, 'hours_per_week': 0.2, 'race': 0.1, 'workclass': 0.0, 'marital_status': 0.0, 'occupation': 0.0}, {'hours_per_week': 1.0, 'education': 0.4, 'workclass': 0.2, 'race': 0.2, 'marital_status': 0.1, 'age': 0.1, 'occupation': 0.0, 'gender': 0.0}, {'marital_status': 0.6, 'occupation': 0.4, 'age': 0.4, 'hours_per_week': 0.3, 'gender': 0.1, 'workclass': 0.0, 'education': 0.0, 'race': 0.0}, {'marital_status': 0.6, 'education': 0.3, 'occupation': 0.3, 'age': 0.3, 'hours_per_week': 0.3, 'workclass': 0.1, 'race': 0.0, 'gender': 0.0}, {'education': 1.0, 'hours_per_week': 0.3, 'workclass': 0.2, 'occupation': 0.2, 'race': 0.2, 'age': 0.1, 'marital_status': 0.0, 'gender': 0.0}, {'age': 0.5, 'hours_per_week': 0.4, 'occupation': 0.2, 'race': 0.2, 'gender': 0.2, 'workclass': 0.1, 'education': 0.1, 'marital_status': 0.1}

In [48]:
cobj = exp.global_feature_importance(x_train[0:50], total_CFs=10, posthoc_sparsity_param=None)
print(cobj.summary_importance)

100%|██████████| 50/50 [00:02<00:00, 17.81it/s]

{'age': 0.432, 'hours_per_week': 0.4, 'marital_status': 0.392, 'education': 0.324, 'occupation': 0.2, 'workclass': 0.138, 'race': 0.082, 'gender': 0.08}


In [49]:
json_str = cobj.to_json()
print(json_str)

{"test_data": [[[50, "Private", "Bachelors", "Married", "Blue-Collar", "White", "Male", 40, 0]], [[50, "Private", "HS-grad", "Married", "Sales", "Other", "Female", 40, 0]], [[71, "Self-Employed", "Some-college", "Separated", "Sales", "Other", "Male", 2, 0]], [[44, "Private", "Prof-school", "Married", "Professional", "White", "Male", 50, 1]], [[55, "Government", "Bachelors", "Married", "White-Collar", "White", "Male", 38, 1]], [[38, "Private", "School", "Married", "Blue-Collar", "White", "Male", 40, 0]], [[38, "Private", "Bachelors", "Married", "White-Collar", "White", "Male", 40, 1]], [[42, "Private", "Some-college", "Divorced", "White-Collar", "White", "Male", 40, 0]], [[33, "Private", "HS-grad", "Single", "Blue-Collar", "Other", "Female", 40, 0]], [[41, "Private", "HS-grad", "Divorced", "Service", "White", "Female", 40, 0]], [[47, "Government", "Some-college", "Married", "Blue-Collar", "White", "Male", 40, 1]], [[69, "Other/Unknown", "HS-grad", "Widowed", "Other/Unknown", "White", "F

In [50]:
imp_r = imp.from_json(json_str)
print([o.visualize_as_dataframe(show_only_changes=True) for o in imp_r.cf_examples_list])
print(imp_r.local_importance)
print(imp_r.summary_importance)

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,50,Private,Bachelors,Married,Blue-Collar,White,Male,40,0



Counterfactual set (new outcome: 1.0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,46.0,-,-,-,-,-,Female,-,1
1,43.0,-,-,-,-,Other,-,-,1
2,-,-,-,-,-,-,-,96.0,1
3,-,-,-,-,-,Other,-,79.0,1
4,-,-,-,Widowed,Sales,-,-,-,1
5,63.0,-,-,-,-,Other,-,-,1
6,59.0,-,-,-,Sales,-,-,-,1
7,-,Self-Employed,-,-,Sales,-,-,-,1
8,-,-,-,-,-,-,-,90.0,1
9,-,-,-,-,-,Other,-,59.0,1


Query instance (original outcome : 0)


...

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,28,Government,Doctorate,Divorced,Professional,White,Female,40,0



Counterfactual set (new outcome: 1.0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,-,Married,-,-,-,18.0,1
1,-,-,-,Married,-,-,-,81.0,1
2,19.0,-,-,Married,-,-,-,-,1
3,-,-,-,Married,-,-,-,38.0,1
4,75.0,-,-,Widowed,-,-,-,-,1
5,-,Other/Unknown,-,Married,-,-,-,-,1
6,27.0,-,-,Married,-,-,-,-,1
7,-,-,Prof-school,Married,-,-,-,-,1
8,77.0,-,-,Separated,-,-,-,-,-
9,73.0,-,-,Married,-,-,-,-,1


[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
[{'age': 0.4, 'race': 0.4, 'hours_per_week': 0.4, 'occupation': 0.3, 'workclass': 0.1, 'marital_status': 0.1, 'gender': 0.1, 'education': 0.0}, {'education': 1.0, 'age': 0.3, 'occupation': 0.2, 'gender': 0.2, 'workclass': 0.1, 'hours_per_week': 0.1, 'marital_status': 0.0, 'race': 0.0}, {'hours_per_week': 0.9, 'education': 0.4, 'race': 0.3, 'workclass': 0.2, 'marital_status': 0.2, 'age': 0.0, 'occupation': 0.0, 'gender': 0.0}, {'age': 0.6, 'marital_status': 0.4, 'occupation': 0.3, 'hours_per_week': 0.3, 'education': 0.2, 'workclass': 0.0, 'race': 0.0, 'gender': 0.0}, {'age': 0.7, 'marital_status': 0.3, 'hours_per_week': 0.2, 'workclass': 0.1, 'education': 0.1, 'occupation': 0.0, 'race': 0.0,